# Microfluidic Emulsion Droplet AI Brain

**Goal:** Build a Colab-ready AI prototype for microfluidic single/double-emulsion droplet analysis, prediction, inverse design, and Streamlit deployment.

This notebook is designed around the uploaded dataset `Comprehensive_normalized.xlsx`, but it also accepts `.csv` files with similar columns. It trains:

1. **Machine-learning regressors** for droplet diameter and generation-rate forecasting.
2. **Deep neural network regressors** for multi-output tabular prediction.
3. **Decision / inverse-design tools** that recommend operating conditions for desired droplet outputs.
4. **A Streamlit prototype app** that can be exported and launched after training.

> Scientific note: in this notebook, “forecasting” means predicting droplet outputs for unseen process conditions. The uploaded data does not appear to contain a time variable, so this is not time-series forecasting unless you add chronological experiment timestamps later.


## 0. Runtime setup

Run this notebook in **Google Colab**. Upload `Comprehensive_normalized.xlsx` or your `.csv` file when prompted.

The cells below avoid hard-coding a Python version. They use the active Colab runtime and print the runtime version for traceability.


In [ ]:
# Runtime and dependency setup
import sys, subprocess, importlib.util, os, json, math, warnings, textwrap, zipfile
from pathlib import Path

print('Python runtime:', sys.version)

# Install only missing packages to keep Colab startup lighter.
required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'joblib': 'joblib',
    'openpyxl': 'openpyxl',
    'streamlit': 'streamlit',
}
missing = [pip_name for module_name, pip_name in required_packages.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
else:
    print('Core packages already available.')

# TensorFlow is usually preinstalled in Colab. Install only if missing.
if importlib.util.find_spec('tensorflow') is None:
    print('TensorFlow not found. Installing tensorflow; this may take several minutes in a fresh environment.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow'])

warnings.filterwarnings('ignore')


In [ ]:
# Imports
import sys, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.inspection import permutation_importance
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_AVAILABLE = True
    print('TensorFlow:', tf.__version__)
except Exception as exc:
    TF_AVAILABLE = False
    print('TensorFlow unavailable:', exc)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
if TF_AVAILABLE:
    tf.keras.utils.set_random_seed(RANDOM_STATE)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)


## 1. Load the microfluidic dataset

This loader accepts either Excel (`.xlsx`, `.xls`) or CSV (`.csv`). In Colab, place the file in `/content`, upload it when prompted, or mount Google Drive and set `DATA_PATH` manually.


In [ ]:
# Data upload / loading
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Change this if your file has a different name or is stored in Google Drive.
DATA_PATH = Path('Comprehensive_normalized.xlsx')

if not DATA_PATH.exists() and Path('/content/Comprehensive_normalized.xlsx').exists():
    DATA_PATH = Path('/content/Comprehensive_normalized.xlsx')

if not DATA_PATH.exists() and IN_COLAB:
    print('Upload your .xlsx or .csv dataset now.')
    uploaded = files.upload()
    if uploaded:
        DATA_PATH = Path(next(iter(uploaded.keys())))

if not DATA_PATH.exists():
    raise FileNotFoundError(
        'Dataset not found. Upload Comprehensive_normalized.xlsx or set DATA_PATH to your file path.'
    )

print('Using data file:', DATA_PATH)

suffix = DATA_PATH.suffix.lower()
if suffix in ['.xlsx', '.xls']:
    df_raw = pd.read_excel(DATA_PATH)
elif suffix == '.csv':
    df_raw = pd.read_csv(DATA_PATH)
else:
    raise ValueError(f'Unsupported file type: {suffix}. Use .xlsx, .xls, or .csv')

print('Raw shape:', df_raw.shape)
df_raw.head()


## 2. Clean columns and audit data quality

The uploaded workbook includes one blank separator column and a trailing empty row. The cleaning step below removes fully empty rows/columns, standardizes duplicate/blank headers, and converts numeric-looking columns to numeric types.


In [ ]:
def clean_column_names(columns):
    cleaned = []
    seen = {}
    for i, col in enumerate(columns):
        name = str(col).strip() if col is not None else ''
        if name == '' or name.lower().startswith('unnamed'):
            name = f'blank_col_{i+1}'
        name = ' '.join(name.split())
        if name in seen:
            seen[name] += 1
            name = f'{name}_{seen[name]}'
        else:
            seen[name] = 0
        cleaned.append(name)
    return cleaned


def clean_dataframe(df):
    df = df.copy()
    df.columns = clean_column_names(df.columns)
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    for col in df.columns:
        if df[col].dtype == 'object':
            converted = pd.to_numeric(df[col], errors='ignore')
            df[col] = converted
    return df.reset_index(drop=True)


df = clean_dataframe(df_raw)
print('Cleaned shape:', df.shape)
print('Columns:')
for c in df.columns:
    print(' -', c)

display(df.head())

audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(dropna=True),
})
display(audit)

# Save a clean copy for reuse.
df.to_csv('microfluidic_cleaned_data.csv', index=False)
print('Saved cleaned CSV: microfluidic_cleaned_data.csv')


## 3. Define targets, inputs, and engineered physics-aware features

The default targets are droplet output variables:

- observed droplet diameter
- normalized droplet diameter
- observed generation rate

The model excludes experiment IDs and all target columns from the input features to reduce leakage.


In [ ]:
def find_columns_by_keywords(columns, keyword_groups):
    matches = []
    for col in columns:
        low = col.lower()
        for group in keyword_groups:
            if all(k.lower() in low for k in group):
                matches.append(col)
                break
    return matches

# Target auto-detection for this dataset and similar droplet-emulsion datasets.
target_keyword_groups = [
    ['observed', 'droplet', 'diameter'],
    ['normalized', 'droplet', 'diameter'],
    ['observed', 'generation', 'rate'],
]
TARGET_COLUMNS = find_columns_by_keywords(df.columns, target_keyword_groups)

# Fallback: ask the user to manually set targets if no target-like columns are found.
if not TARGET_COLUMNS:
    raise ValueError(
        'No default target columns found. Manually set TARGET_COLUMNS to droplet output columns.'
    )

ID_COLUMNS = [c for c in df.columns if c.lower() in ['experiment', 'experiment id', 'id', 'index']]

print('Target columns:', TARGET_COLUMNS)
print('ID columns excluded from modeling:', ID_COLUMNS)


def add_engineered_features(data, feature_source_columns):
    out = data.copy()
    cols = set(out.columns)

    def safe_div(a, b):
        return np.where(np.abs(b) > 1e-12, a / b, np.nan)

    # Common microfluidic interactions. They are added only when source columns exist.
    if {'Flow rate ratio', 'Capillary number'}.issubset(cols):
        out['eng_Ca_x_flow_ratio'] = out['Capillary number'] * out['Flow rate ratio']
        out['eng_Ca_div_flow_ratio'] = safe_div(out['Capillary number'], out['Flow rate ratio'])

    if {'Orifice width (um)', 'Hydraulic diameter'}.issubset(cols):
        out['eng_orifice_div_hydraulic_diameter'] = safe_div(out['Orifice width (um)'], out['Hydraulic diameter'])

    if {'Orifice width (um)', 'Channel Depth'}.issubset(cols):
        out['eng_orifice_area_proxy'] = out['Orifice width (um)'] * out['Channel Depth']
        out['eng_channel_depth_div_orifice'] = safe_div(out['Channel Depth'], out['Orifice width (um)'])

    if {'Normalized continuous inlet', 'Normalized dispersed inlet'}.issubset(cols):
        out['eng_continuous_div_dispersed_inlet'] = safe_div(
            out['Normalized continuous inlet'], out['Normalized dispersed inlet']
        )

    if {'viscosity ratio', 'Capillary number'}.issubset(cols):
        out['eng_viscosity_x_Ca'] = out['viscosity ratio'] * out['Capillary number']

    if {'Dispersed flow rate ul/h', 'Flow rate ratio'}.issubset(cols):
        out['eng_est_continuous_flow_rate_ul_h'] = out['Dispersed flow rate ul/h'] * out['Flow rate ratio']

    # Replace infinite values created by divisions.
    out = out.replace([np.inf, -np.inf], np.nan)
    return out

# Use numeric input columns only for this first production prototype.
raw_numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
base_feature_cols = [c for c in raw_numeric_cols if c not in TARGET_COLUMNS + ID_COLUMNS]
BASE_FEATURE_COLUMNS = base_feature_cols.copy()

df_features = add_engineered_features(df, base_feature_cols)
all_numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
FEATURE_COLUMNS = [c for c in all_numeric_cols if c not in TARGET_COLUMNS + ID_COLUMNS]

model_df = df_features[FEATURE_COLUMNS + TARGET_COLUMNS].dropna(subset=TARGET_COLUMNS).copy()

print(f'Number of base input features: {len(BASE_FEATURE_COLUMNS)}')
print(BASE_FEATURE_COLUMNS)
print(f'Number of total model features after engineering: {len(FEATURE_COLUMNS)}')
print(FEATURE_COLUMNS)
print(f'Number of targets: {len(TARGET_COLUMNS)}')
print(TARGET_COLUMNS)
print('Modeling rows before feature imputation:', model_df.shape[0])


## 4. Exploratory analysis

These plots show the response distributions and quick feature-target relationships. Use them to detect outliers, nonlinearities, and possible regions where single/double-emulsion behavior differs.


In [ ]:
# Summary statistics
summary = model_df[FEATURE_COLUMNS + TARGET_COLUMNS].describe().T
summary['missing'] = model_df[FEATURE_COLUMNS + TARGET_COLUMNS].isna().sum()
display(summary)

# Target distributions
for target in TARGET_COLUMNS:
    plt.figure(figsize=(7, 4))
    plt.hist(model_df[target].dropna(), bins=35)
    plt.title(f'Distribution: {target}')
    plt.xlabel(target)
    plt.ylabel('Count')
    plt.show()

# Correlation heatmap using matplotlib only
corr_cols = FEATURE_COLUMNS + TARGET_COLUMNS
corr = model_df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(max(10, len(corr_cols) * 0.45), max(8, len(corr_cols) * 0.45)))
plt.imshow(corr, aspect='auto')
plt.colorbar(label='Pearson correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title('Feature / Target Correlation Map')
plt.tight_layout()
plt.show()

# Top absolute correlations with each target
for target in TARGET_COLUMNS:
    s = corr[target].drop(labels=TARGET_COLUMNS, errors='ignore').abs().sort_values(ascending=False).head(10)
    print('\nTop feature correlations with', target)
    display(s.to_frame('abs_corr'))


## 5. Train/test split and metric functions

The dataset is tabular and relatively small, so the notebook uses a reproducible holdout test split plus cross-validation for the selected model.


In [ ]:
X = model_df[FEATURE_COLUMNS].copy()
y = model_df[TARGET_COLUMNS].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print('Train shape:', X_train.shape, y_train.shape)
print('Test shape:', X_test.shape, y_test.shape)


def regression_metrics(y_true, y_pred, target_names):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rows = []
    for i, target in enumerate(target_names):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        rmse = float(np.sqrt(mean_squared_error(yt, yp)))
        mae = float(mean_absolute_error(yt, yp))
        r2 = float(r2_score(yt, yp))
        mape = float(np.mean(np.abs(yt - yp) / np.maximum(np.abs(yt), 1e-9)) * 100)
        rows.append({
            'target': target,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2,
            'MAPE_pct': mape,
        })
    return pd.DataFrame(rows)


def summarize_model_metrics(name, metrics_df):
    return {
        'model': name,
        'mean_R2': metrics_df['R2'].mean(),
        'mean_RMSE': metrics_df['RMSE'].mean(),
        'mean_MAE': metrics_df['MAE'].mean(),
        'mean_MAPE_pct': metrics_df['MAPE_pct'].mean(),
    }


## 6. Machine-learning models

The training suite compares a simple baseline to stronger nonlinear models. The target values are modeled using a `log1p` transform because droplet diameter and generation rate are positive and skewed.


In [ ]:
# Common preprocessing for numeric tabular data
numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[('num', numeric_preprocess, FEATURE_COLUMNS)],
    remainder='drop',
    verbose_feature_names_out=False,
)


def positive_target_wrapper(regressor):
    """Use log1p target transform and invert predictions back to original units."""
    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )

model_candidates = {
    'DummyMedian': Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', DummyRegressor(strategy='median')),
    ]),
    'Ridge': positive_target_wrapper(Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ])),
    'RandomForest': positive_target_wrapper(Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])),
    'ExtraTrees': positive_target_wrapper(Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', ExtraTreesRegressor(
            n_estimators=400,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])),
}

# HistGradientBoosting can be useful, but it is optional because some free runtimes
# are slower with nested multi-output boosting.
RUN_ADVANCED_MODELS = False
if RUN_ADVANCED_MODELS:
    model_candidates['HistGradientBoosting'] = positive_target_wrapper(Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', MultiOutputRegressor(HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.04,
            max_leaf_nodes=31,
            random_state=RANDOM_STATE,
        ))),
    ]))

fitted_models = {}
metric_frames = []
summary_rows = []

for name, model in model_candidates.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    pred = np.clip(model.predict(X_test), 0, None)
    metrics = regression_metrics(y_test.values, pred, TARGET_COLUMNS)
    metrics.insert(0, 'model', name)
    metric_frames.append(metrics)
    summary_rows.append(summarize_model_metrics(name, metrics))
    fitted_models[name] = model

metrics_all = pd.concat(metric_frames, ignore_index=True)
model_summary = pd.DataFrame(summary_rows).sort_values(['mean_R2', 'mean_RMSE'], ascending=[False, True])

display(model_summary)
display(metrics_all)

BEST_MODEL_NAME = model_summary.iloc[0]['model']
best_model = fitted_models[BEST_MODEL_NAME]
print('Best holdout model:', BEST_MODEL_NAME)


## 7. Hyperparameter tuning for the industrial ML model

The cell below tunes an ExtraTrees model. Keep `RUN_TUNING=True` for better accuracy; set it to `False` for a faster demo.


In [ ]:
RUN_TUNING = False

if RUN_TUNING:
    tuning_model = positive_target_wrapper(Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ]))

    param_distributions = {
        'regressor__model__n_estimators': [200, 300, 500, 700],
        'regressor__model__max_depth': [None, 6, 10, 14, 20],
        'regressor__model__min_samples_leaf': [1, 2, 3, 5],
        'regressor__model__max_features': [0.6, 0.8, 1.0, 'sqrt'],
    }

    search = RandomizedSearchCV(
        estimator=tuning_model,
        param_distributions=param_distributions,
        n_iter=12,
        scoring='neg_root_mean_squared_error',
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=1,
    )
    search.fit(X_train, y_train)
    tuned_model = search.best_estimator_
    tuned_pred = np.clip(tuned_model.predict(X_test), 0, None)
    tuned_metrics = regression_metrics(y_test.values, tuned_pred, TARGET_COLUMNS)
    tuned_summary = summarize_model_metrics('TunedExtraTrees', tuned_metrics)

    print('Best tuning params:', search.best_params_)
    display(pd.DataFrame([tuned_summary]))
    display(tuned_metrics)

    # Replace best model if tuning improves mean R2.
    if tuned_summary['mean_R2'] >= model_summary.iloc[0]['mean_R2']:
        BEST_MODEL_NAME = 'TunedExtraTrees'
        best_model = tuned_model
        fitted_models[BEST_MODEL_NAME] = best_model
        print('Tuned model selected as best_model.')
    else:
        print('Original holdout best model retained:', BEST_MODEL_NAME)
else:
    print('Skipping tuning. Current best_model:', BEST_MODEL_NAME)


## 8. Cross-validation and error analysis

Use cross-validation to estimate generalization stability. Then inspect which outputs have the largest errors.


In [ ]:
# Cross-validation on the selected model using averaged RMSE across targets.
cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    best_model,
    X,
    y,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=1,
)
print('Selected model:', BEST_MODEL_NAME)
print('CV RMSE per fold:', -cv_scores)
print('CV RMSE mean +/- std:', (-cv_scores).mean(), '+/-', (-cv_scores).std())

# Holdout predictions and residuals
best_pred = np.clip(best_model.predict(X_test), 0, None)
best_metrics = regression_metrics(y_test.values, best_pred, TARGET_COLUMNS)
display(best_metrics)

pred_df = pd.DataFrame(best_pred, columns=[f'pred_{c}' for c in TARGET_COLUMNS], index=y_test.index)
resid_df = y_test.copy()
for target in TARGET_COLUMNS:
    resid_df[f'pred_{target}'] = pred_df[f'pred_{target}']
    resid_df[f'abs_error_{target}'] = np.abs(resid_df[target] - resid_df[f'pred_{target}'])

display(resid_df.sort_values(f'abs_error_{TARGET_COLUMNS[0]}', ascending=False).head(10))

for target in TARGET_COLUMNS:
    plt.figure(figsize=(5.5, 5))
    plt.scatter(y_test[target], pred_df[f'pred_{target}'], alpha=0.75)
    lo = min(y_test[target].min(), pred_df[f'pred_{target}'].min())
    hi = max(y_test[target].max(), pred_df[f'pred_{target}'].max())
    plt.plot([lo, hi], [lo, hi], linestyle='--')
    plt.xlabel(f'Actual {target}')
    plt.ylabel(f'Predicted {target}')
    plt.title(f'Actual vs Predicted: {target}')
    plt.tight_layout()
    plt.show()


## 9. Explainability: feature importance

For tree models, impurity-based importance is shown. A model-agnostic permutation importance is also computed on the holdout set.


In [ ]:
# Try tree-based feature importances first.
def get_fitted_pipeline_from_target_wrapper(model):
    return getattr(model, 'regressor_', model)

fitted_pipeline = get_fitted_pipeline_from_target_wrapper(best_model)
model_step = fitted_pipeline.named_steps.get('model') if hasattr(fitted_pipeline, 'named_steps') else None

importance_df = None
if hasattr(model_step, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': FEATURE_COLUMNS,
        'importance': model_step.feature_importances_,
    }).sort_values('importance', ascending=False)
    print('Tree-based feature importance:')
    display(importance_df.head(20))

if importance_df is None:
    print('Tree importances unavailable; using permutation importance.')

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring='r2',
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
perm_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

display(perm_df.head(20))

plt.figure(figsize=(8, 5))
top_perm = perm_df.head(12).iloc[::-1]
plt.barh(top_perm['feature'], top_perm['importance_mean'])
plt.xlabel('Permutation importance, mean R2 drop')
plt.title('Top predictive drivers')
plt.tight_layout()
plt.show()


## 10. Deep neural network for multi-output droplet prediction

Tree models often perform very well on small tabular engineering datasets. The neural network below becomes more valuable as you add larger, more diverse industrial datasets, image-derived features, or sensor time-series inputs.


In [ ]:
RUN_DNN = True

if RUN_DNN and TF_AVAILABLE:
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_nn = x_scaler.fit_transform(SimpleImputer(strategy='median').fit_transform(X_train))
    X_test_nn = x_scaler.transform(SimpleImputer(strategy='median').fit(X_train).transform(X_test))

    y_train_log = np.log1p(y_train.values)
    y_test_log = np.log1p(y_test.values)
    y_train_nn = y_scaler.fit_transform(y_train_log)
    y_test_nn = y_scaler.transform(y_test_log)

    dnn_model = keras.Sequential([
        layers.Input(shape=(X_train_nn.shape[1],)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.15),
        layers.Dense(96, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.10),
        layers.Dense(64, activation='relu'),
        layers.Dense(len(TARGET_COLUMNS), activation='linear'),
    ])

    dnn_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='mse',
        metrics=[keras.metrics.RootMeanSquaredError(name='rmse')],
    )

    callbacks = [
        keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True, monitor='val_loss'),
        keras.callbacks.ReduceLROnPlateau(patience=12, factor=0.5, min_lr=1e-5, monitor='val_loss'),
    ]

    history = dnn_model.fit(
        X_train_nn,
        y_train_nn,
        validation_split=0.20,
        epochs=300,
        batch_size=32,
        callbacks=callbacks,
        verbose=0,
    )

    dnn_pred_scaled = dnn_model.predict(X_test_nn, verbose=0)
    dnn_pred_log = y_scaler.inverse_transform(dnn_pred_scaled)
    dnn_pred = np.clip(np.expm1(dnn_pred_log), 0, None)
    dnn_metrics = regression_metrics(y_test.values, dnn_pred, TARGET_COLUMNS)
    display(dnn_metrics)
    print('DNN mean R2:', dnn_metrics['R2'].mean())

    plt.figure(figsize=(7, 4))
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE on scaled log targets')
    plt.title('DNN training curve')
    plt.legend()
    plt.tight_layout()
    plt.show()

    dnn_model.save('microfluidic_dnn.keras')
    joblib.dump({
        'x_scaler': x_scaler,
        'y_scaler': y_scaler,
        'feature_imputer': SimpleImputer(strategy='median').fit(X_train),
        'feature_columns': FEATURE_COLUMNS,
        'target_columns': TARGET_COLUMNS,
    }, 'microfluidic_dnn_preprocessors.joblib')
    print('Saved DNN artifacts.')
else:
    print('Skipping DNN. Set RUN_DNN=True and ensure TensorFlow is available.')


## 11. Regime discovery or emulsion-type classification

If your future data contains a labeled column such as `emulsion type`, `single/double`, or `regime`, this section can be extended into a supervised classifier. In the current workbook, no explicit type label is visible, so the cell below performs unsupervised clustering to discover operating regimes.


In [ ]:
label_candidates = [c for c in df.columns if any(k in c.lower() for k in ['emulsion', 'type', 'regime', 'class', 'label'])]
print('Possible label columns:', label_candidates)

if label_candidates:
    print('A potential label column exists. For production, train a classifier using that label after confirming its meaning.')
else:
    cluster_input = model_df[FEATURE_COLUMNS + TARGET_COLUMNS].copy()
    cluster_input = pd.DataFrame(
        SimpleImputer(strategy='median').fit_transform(cluster_input),
        columns=cluster_input.columns,
    )
    cluster_scaled = StandardScaler().fit_transform(cluster_input)

    sil_scores = {}
    for k in range(2, 8):
        labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit_predict(cluster_scaled)
        sil_scores[k] = silhouette_score(cluster_scaled, labels)

    best_k = max(sil_scores, key=sil_scores.get)
    kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
    model_df['DiscoveredRegime'] = kmeans.fit_predict(cluster_scaled)

    print('Silhouette scores:', sil_scores)
    print('Selected number of discovered regimes:', best_k)
    display(model_df.groupby('DiscoveredRegime')[TARGET_COLUMNS].agg(['count', 'mean', 'median']).round(3))


## 12. Decision layer: inverse design for target droplet outputs

This is the product-oriented “AI brain” layer. Given desired droplet outputs, it samples feasible operating conditions inside the training-data domain, predicts outputs, scores candidates, and recommends the best operating settings.

For regulated medical/biological use, treat these as **AI suggestions** that must be verified experimentally.


In [ ]:
# Metadata for feature ranges and discrete-like variables.
# We keep profiles for both base laboratory inputs and engineered model features.
base_feature_profile = {}
for col in BASE_FEATURE_COLUMNS:
    s = df_features[col].dropna()
    unique_vals = np.sort(s.unique())
    base_feature_profile[col] = {
        'min': float(s.min()),
        'median': float(s.median()),
        'max': float(s.max()),
        'n_unique': int(s.nunique()),
        'unique_values': unique_vals.astype(float).tolist() if s.nunique() <= 30 else None,
    }

feature_profile = {}
for col in FEATURE_COLUMNS:
    s = model_df[col].dropna()
    unique_vals = np.sort(s.unique())
    feature_profile[col] = {
        'min': float(s.min()),
        'median': float(s.median()),
        'max': float(s.max()),
        'n_unique': int(s.nunique()),
        'unique_values': unique_vals.astype(float).tolist() if s.nunique() <= 30 else None,
    }

target_profile = {}
for col in TARGET_COLUMNS:
    s = model_df[col].dropna()
    target_profile[col] = {
        'min': float(s.min()),
        'median': float(s.median()),
        'max': float(s.max()),
    }


def sample_candidate_space(n_candidates=20000, random_state=RANDOM_STATE):
    # Sample physically meaningful base inputs, then derive engineered features consistently.
    rng = np.random.default_rng(random_state)
    sampled = {}
    for col, profile in base_feature_profile.items():
        if profile['unique_values'] is not None:
            sampled[col] = rng.choice(profile['unique_values'], size=n_candidates)
        else:
            sampled[col] = rng.uniform(profile['min'], profile['max'], size=n_candidates)
    base_candidates = pd.DataFrame(sampled)[BASE_FEATURE_COLUMNS]
    engineered_candidates = add_engineered_features(base_candidates, BASE_FEATURE_COLUMNS)
    for col in FEATURE_COLUMNS:
        if col not in engineered_candidates.columns:
            engineered_candidates[col] = feature_profile[col]['median']
    return engineered_candidates[FEATURE_COLUMNS]


def recommend_operating_conditions(
    desired_targets,
    model=None,
    n_candidates=10000,
    top_k=10,
    weights=None,
    random_state=RANDOM_STATE,
):
    if model is None:
        model = best_model
    if weights is None:
        weights = {target: 1.0 for target in TARGET_COLUMNS}

    candidates = sample_candidate_space(n_candidates=n_candidates, random_state=random_state)
    preds = np.clip(model.predict(candidates), 0, None)
    pred_df = pd.DataFrame(preds, columns=TARGET_COLUMNS)

    score = np.zeros(len(candidates))
    for target in TARGET_COLUMNS:
        desired = float(desired_targets[target])
        denom = max(abs(desired), 1e-9)
        score += weights.get(target, 1.0) * ((pred_df[target] - desired) / denom) ** 2
    score = np.sqrt(score)

    result = pd.concat(
        [candidates.reset_index(drop=True), pred_df.add_prefix('pred_')],
        axis=1,
    )
    result['decision_score_lower_is_better'] = score
    return result.sort_values('decision_score_lower_is_better').head(top_k).reset_index(drop=True)

# Example desired target: median observed outputs from the dataset.
desired = {target: target_profile[target]['median'] for target in TARGET_COLUMNS}
print('Example desired targets:', desired)
recommendations = recommend_operating_conditions(desired, top_k=10)
display(recommendations)


## 13. Uncertainty and out-of-domain safety checks

Industrial and medical users need reliability flags. The simple checks below warn when a requested operating condition is outside the training domain and estimate nearest-neighbor distance from known experiments.


In [ ]:
# Fit nearest-neighbor distance model in the model's preprocessed feature space.
fitted_pipeline = get_fitted_pipeline_from_target_wrapper(best_model)
preprocess_step = fitted_pipeline.named_steps['preprocess']
X_train_processed = preprocess_step.transform(X_train)

nn_model = NearestNeighbors(n_neighbors=min(6, len(X_train_processed))).fit(X_train_processed)
train_neighbor_distances = nn_model.kneighbors(X_train_processed)[0]
# Exclude self-distance in column 0 if present.
train_mean_distance = train_neighbor_distances[:, 1:].mean(axis=1) if train_neighbor_distances.shape[1] > 1 else train_neighbor_distances.mean(axis=1)
OOD_DISTANCE_THRESHOLD = float(np.quantile(train_mean_distance, 0.95))
print('95th percentile training-neighborhood distance:', OOD_DISTANCE_THRESHOLD)


def check_domain(row_df):
    warnings_out = []
    for col, profile in feature_profile.items():
        value = float(row_df.iloc[0][col])
        if value < profile['min'] or value > profile['max']:
            warnings_out.append(f'{col}={value:.4g} is outside training range [{profile["min"]:.4g}, {profile["max"]:.4g}]')

    processed = preprocess_step.transform(row_df[FEATURE_COLUMNS])
    dist = float(nn_model.kneighbors(processed)[0].mean())
    if dist > OOD_DISTANCE_THRESHOLD:
        warnings_out.append(f'Nearest-neighbor distance {dist:.4g} exceeds training 95% threshold {OOD_DISTANCE_THRESHOLD:.4g}')
    return dist, warnings_out

example_row = X_test.iloc[[0]].copy()
example_pred = pd.DataFrame(np.clip(best_model.predict(example_row), 0, None), columns=TARGET_COLUMNS)
dist, warnings_out = check_domain(example_row)
print('Example prediction:')
display(pd.concat([example_row.reset_index(drop=True), example_pred.add_prefix('pred_')], axis=1))
print('OOD distance:', dist)
print('Warnings:', warnings_out if warnings_out else 'None')


## 14. Save the trained model and metadata

The saved scikit-learn model can be loaded by Streamlit, Flask, FastAPI, or a batch pipeline. The metadata JSON stores feature names, target names, training ranges, and metrics.


In [ ]:
# Fit the selected best model on the full cleaned dataset for deployment.
final_model = best_model
final_model.fit(X, y)

metadata = {
    'project': 'Microfluidic Emulsion Droplet AI Brain',
    'data_file': str(DATA_PATH),
    'n_rows_cleaned': int(df.shape[0]),
    'base_feature_columns': BASE_FEATURE_COLUMNS,
    'feature_columns': FEATURE_COLUMNS,
    'target_columns': TARGET_COLUMNS,
    'best_model_name': BEST_MODEL_NAME,
    'holdout_metrics': best_metrics.to_dict(orient='records'),
    'base_feature_profile': base_feature_profile,
    'feature_profile': feature_profile,
    'target_profile': target_profile,
    'ood_distance_threshold': OOD_DISTANCE_THRESHOLD,
    'notes': [
        'Predictions are valid primarily inside the training-data domain.',
        'Use wet-lab validation before medical, biological, or industrial decisions.',
        'Forecasting here means tabular prediction for new process conditions, not time-series forecasting.',
    ],
}

joblib.dump(final_model, 'microfluidic_best_model.joblib')
with open('microfluidic_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved: microfluidic_best_model.joblib')
print('Saved: microfluidic_model_metadata.json')


## 15. Generate a Streamlit prototype app

After training, this cell writes a production-style `streamlit_app.py`. The app lets users enter process settings, get predicted droplet outputs, see domain warnings, and run inverse-design recommendations.


In [ ]:
streamlit_code = r"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import streamlit as st
import joblib

st.set_page_config(page_title='Microfluidic Droplet AI', layout='wide')

MODEL_PATH = Path('microfluidic_best_model.joblib')
META_PATH = Path('microfluidic_model_metadata.json')

@st.cache_resource
def load_assets():
    model = joblib.load(MODEL_PATH)
    metadata = json.loads(META_PATH.read_text())
    return model, metadata

model, metadata = load_assets()
base_feature_columns = metadata.get('base_feature_columns', metadata['feature_columns'])
feature_columns = metadata['feature_columns']
target_columns = metadata['target_columns']
base_feature_profile = metadata.get('base_feature_profile', metadata['feature_profile'])
feature_profile = metadata['feature_profile']
target_profile = metadata['target_profile']

def safe_div(a, b):
    return np.where(np.abs(b) > 1e-12, a / b, np.nan)

def add_engineered_features_app(data):
    out = data.copy()
    cols = set(out.columns)
    if {'Flow rate ratio', 'Capillary number'}.issubset(cols):
        out['eng_Ca_x_flow_ratio'] = out['Capillary number'] * out['Flow rate ratio']
        out['eng_Ca_div_flow_ratio'] = safe_div(out['Capillary number'], out['Flow rate ratio'])
    if {'Orifice width (um)', 'Hydraulic diameter'}.issubset(cols):
        out['eng_orifice_div_hydraulic_diameter'] = safe_div(out['Orifice width (um)'], out['Hydraulic diameter'])
    if {'Orifice width (um)', 'Channel Depth'}.issubset(cols):
        out['eng_orifice_area_proxy'] = out['Orifice width (um)'] * out['Channel Depth']
        out['eng_channel_depth_div_orifice'] = safe_div(out['Channel Depth'], out['Orifice width (um)'])
    if {'Normalized continuous inlet', 'Normalized dispersed inlet'}.issubset(cols):
        out['eng_continuous_div_dispersed_inlet'] = safe_div(out['Normalized continuous inlet'], out['Normalized dispersed inlet'])
    if {'viscosity ratio', 'Capillary number'}.issubset(cols):
        out['eng_viscosity_x_Ca'] = out['viscosity ratio'] * out['Capillary number']
    if {'Dispersed flow rate ul/h', 'Flow rate ratio'}.issubset(cols):
        out['eng_est_continuous_flow_rate_ul_h'] = out['Dispersed flow rate ul/h'] * out['Flow rate ratio']
    out = out.replace([np.inf, -np.inf], np.nan)
    for col in feature_columns:
        if col not in out.columns:
            out[col] = float(feature_profile[col]['median'])
    return out[feature_columns]

st.title('Microfluidic Emulsion Droplet AI')
st.caption('Predict droplet diameter and generation rate, then recommend operating conditions for target outputs.')

with st.expander('Model information', expanded=False):
    st.write('Best model:', metadata.get('best_model_name', 'unknown'))
    st.write('Training rows:', metadata.get('n_rows_cleaned'))
    st.dataframe(pd.DataFrame(metadata.get('holdout_metrics', [])))
    st.warning('Use predictions inside the training-data domain and validate experimentally before regulated or production decisions.')

st.sidebar.header('Laboratory / process inputs')
input_values = {}
for col in base_feature_columns:
    p = base_feature_profile[col]
    min_v, med_v, max_v = float(p['min']), float(p['median']), float(p['max'])
    if p.get('unique_values') and len(p['unique_values']) <= 30:
        options = [float(v) for v in p['unique_values']]
        default_idx = min(range(len(options)), key=lambda i: abs(options[i] - med_v))
        input_values[col] = st.sidebar.selectbox(col, options=options, index=default_idx)
    else:
        input_values[col] = st.sidebar.number_input(col, value=med_v, min_value=min_v, max_value=max_v)

base_input_df = pd.DataFrame([input_values])[base_feature_columns]
model_input_df = add_engineered_features_app(base_input_df)

col1, col2 = st.columns([1, 1])
with col1:
    st.subheader('Single-condition prediction')
    if st.button('Predict droplet outputs', type='primary'):
        pred = np.clip(model.predict(model_input_df), 0, None)
        pred_df = pd.DataFrame(pred, columns=target_columns)
        st.dataframe(pred_df, use_container_width=True)

        warnings = []
        for col in base_feature_columns:
            p = base_feature_profile[col]
            v = float(base_input_df.iloc[0][col])
            if v < float(p['min']) or v > float(p['max']):
                warnings.append(f'{col}={v:.4g} is outside training range [{float(p["min"]):.4g}, {float(p["max"]):.4g}]')
        if warnings:
            st.warning('\n'.join(warnings))
        else:
            st.success('All base inputs are inside the observed training ranges.')

with col2:
    st.subheader('Inverse design recommendations')
    desired = {}
    for target in target_columns:
        p = target_profile[target]
        desired[target] = st.number_input(
            f'Desired {target}',
            value=float(p['median']),
            min_value=float(p['min']),
            max_value=float(p['max']),
        )

    n_candidates = st.slider('Candidate search size', min_value=1000, max_value=50000, value=10000, step=1000)
    top_k = st.slider('Number of recommendations', min_value=3, max_value=25, value=10)

    def sample_candidate_space(n_candidates, seed=42):
        rng = np.random.default_rng(seed)
        sampled = {}
        for col in base_feature_columns:
            p = base_feature_profile[col]
            if p.get('unique_values') and len(p['unique_values']) <= 30:
                sampled[col] = rng.choice([float(v) for v in p['unique_values']], size=n_candidates)
            else:
                sampled[col] = rng.uniform(float(p['min']), float(p['max']), size=n_candidates)
        base_candidates = pd.DataFrame(sampled)[base_feature_columns]
        model_candidates = add_engineered_features_app(base_candidates)
        return base_candidates, model_candidates

    if st.button('Recommend operating conditions'):
        base_candidates, model_candidates = sample_candidate_space(n_candidates)
        preds = np.clip(model.predict(model_candidates), 0, None)
        pred_df = pd.DataFrame(preds, columns=target_columns)
        score = np.zeros(len(base_candidates))
        for target in target_columns:
            denom = max(abs(float(desired[target])), 1e-9)
            score += ((pred_df[target] - float(desired[target])) / denom) ** 2
        result = pd.concat([base_candidates.reset_index(drop=True), pred_df.add_prefix('pred_')], axis=1)
        result['decision_score_lower_is_better'] = np.sqrt(score)
        result = result.sort_values('decision_score_lower_is_better').head(top_k).reset_index(drop=True)
        st.dataframe(result, use_container_width=True)
        st.download_button('Download recommendations CSV', result.to_csv(index=False), file_name='microfluidic_recommendations.csv')
"""

Path('streamlit_app.py').write_text(streamlit_code)
print('Saved: streamlit_app.py')
print('Preview first 40 lines:')
print('\n'.join(streamlit_code.splitlines()[:40]))


## 16. Run Streamlit from Colab

Colab does not expose local ports directly. The cell below starts Streamlit and then uses `localtunnel` through `npx` to create a temporary public URL.

Run this only after the model and metadata have been saved.


In [ ]:
RUN_STREAMLIT = False

if RUN_STREAMLIT:
    # Start Streamlit in the background.
    import subprocess, time
    process = subprocess.Popen([
        sys.executable, '-m', 'streamlit', 'run', 'streamlit_app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
    ])
    time.sleep(5)
    print('Streamlit started on port 8501.')
    print('Now create a public URL with localtunnel:')
    print('Run this in a new Colab cell:')
    print('!npx localtunnel --port 8501')
else:
    print('Streamlit launch skipped. Set RUN_STREAMLIT=True when you are ready to demo the app.')


## 17. Package artifacts for download

This creates a zip file containing the trained model, metadata, cleaned data, Streamlit app, and optional neural network artifacts.


In [ ]:
artifact_files = [
    'microfluidic_best_model.joblib',
    'microfluidic_model_metadata.json',
    'microfluidic_cleaned_data.csv',
    'streamlit_app.py',
]
optional_files = [
    'microfluidic_dnn.keras',
    'microfluidic_dnn_preprocessors.joblib',
]
artifact_files += [f for f in optional_files if Path(f).exists()]

zip_path = Path('microfluidic_ai_product_artifacts.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for f in artifact_files:
        if Path(f).exists():
            z.write(f)
            print('Added:', f)

print('Created:', zip_path)

if IN_COLAB:
    print('Use the Files panel to download the zip, or uncomment the next two lines:')
    print('# from google.colab import files')
    print('# files.download(str(zip_path))')


## 18. Product roadmap for industry/medical positioning

To turn this prototype into a serious product for biotechnology and microfluidics companies:

1. **Data expansion:** add more experiments, explicit single/double-emulsion labels, fluid chemistry, surfactant concentration, temperature, imaging-derived features, chip material, and batch/lot metadata.
2. **Validation:** keep a locked test set by chip family and date; report uncertainty, out-of-domain flags, and calibration curves.
3. **Decision support:** optimize for target diameter, generation rate, monodispersity, stability, material cost, and throughput.
4. **Human review:** keep a scientist-in-the-loop workflow for medical/biological workflows.
5. **Deployment:** Streamlit for demos; FastAPI + database + model registry for production; audit logs for regulated customers.

This notebook gives you the first working AI brain: prediction, deep learning, inverse design, validation metrics, explainability, and a web-app prototype.
